# 解析单细胞转录组数据中的细胞间通讯


## 安装所需库


In [ ]:
# Download a shell script to add CRAN repositories to Ubuntu Jammy
download.file("https://github.com/eddelbuettel/r2u/raw/master/inst/scripts/add_cranapt_jammy.sh",
              "add_cranapt_jammy.sh")

# Change file permissions to make it executable
Sys.chmod("add_cranapt_jammy.sh", "0755")

# Run the shell script
system("./add_cranapt_jammy.sh")

# Enable the binary package manager (bspm) to handle system and CRAN packages
bspm::enable()
options(bspm.version.check = FALSE)


我们将创建一个 R 函数，用于执行系统调用。


In [ ]:
# Define a function to execute shell commands and print the output
shell_call <- function(command, ...) {
  result <- system(command, intern = TRUE, ...)
  cat(paste0(result, collapse = "\n"))
}

安装所需库。


In [ ]:
# Install required R packages
install.packages("R.utils")

# Install Seurat Wrappers from GitHub (commented)
# remotes::install_github('satijalab/seurat-wrappers@d28512f804d5fe05e6d68900ca9221020d52cf1d', upgrade=F)

# Install BiocManager if not already installed
if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager", quiet = T)

# Install Harmony and LIANA from GitHub
install.packages("harmony")
remotes::install_github('saezlab/liana', upgrade=F)


In [ ]:
# Install the Seurat package for single-cell analysis
remotes::install_version("Seurat", version = "4.4.0")
remotes::install_version("SeuratObject", version = "4.1.4")


In [ ]:
BiocManager::install(c("sparseMatrixStats", "DelayedMatrixStats"))

## 引言


LIANA（Ligand-Receptor Inference Analysis）利用已有先验知识，提供多种统计方法，从单细胞转录组数据中推断 ligand-receptor（配体-受体）相互作用。本 notebook 的目标是演示如何在感兴趣的数据上以基础方式使用 LIANA。

使用 LIANA 的步骤：
1. 数据准备：
* 上传格式合适的单细胞转录组数据。
* 确保数据已经正确归一化并完成注释。

2. LIANA 安装和导入：
* 如果尚未安装 LIANA 包，可使用 `pip install liana` 命令安装。
* 在 notebook 中用 `import liana` 导入该包。

3. LIANA 配置：
* 设置分析所需参数，例如感兴趣的细胞类型和信号通路。
* 使用 LIANA 的专用函数定义希望研究的配体-受体相互作用。

4. 执行分析：
* 使用 LIANA 中可用的统计方法执行分析。
* 分析结果，识别样本中显著的配体-受体相互作用。

5. 结果解释：
* 使用内置可视化工具展示结果，或导出数据用于进一步分析。
* 在研究的生物学背景下解释识别出的相互作用。

![LIANA](https://saezlab.github.io/liana/articles/ligrec_pipe.png)


In [ ]:
# Load necessary libraries for data manipulation and analysis
library(tidyverse)  # Collection of packages for data science
library(magrittr)   # Pipe operator
library(liana)      # Cell-cell communication analysis
library(Seurat)     # Single-cell RNA-seq analysis

In [ ]:
#sessionInfo()

在本节中，我们将展示 LIANA 从多种工具中整合实现的所有方法。每种方法都基于不同假设推断相关的 ligand-receptor 相互作用。通常，每种方法都会为每个 ligand-receptor 配对返回两个评分：

1. **Magnitude（Strength）Score：** 表示相互作用强度。
2. **Specificity Score：** 表示该相互作用对某一对细胞身份的特异性。


In [ ]:
# Show available methods in the current R session
show_methods()

不同的 ligand-receptor 相互作用资源可以在这里找到。`consensus` 会整合所有其他资源。


In [ ]:
# Show available resources in the current R session
show_resources()

## 加载数据


这里加载用于研究细胞间通讯的目标数据。


In [ ]:
# Download the COVID-19 dataset (RDS file) from Dropbox
download.file("https://www.dropbox.com/scl/fi/1ysew52kr8o2riahzubcw/BALF-COVID19-Liao_et_al-NatMed-2020.rds?rlkey=tg3tpn8la6oth25wvx3a22qt9&dl=1", "COVID.rds")

In [ ]:
# Read the RDS file into a variable called 'testdata'
testdata <- readRDS('COVID.rds')

In [ ]:
# Display an overview of the 'testdata' structure
testdata %>% dplyr::glimpse()

In [ ]:
# This specific condition 'group == "S" ' is being used to filter the rows. Only those rows where the group column value is equal to "S" will be included in the subset.
testdata <- subset(x = testdata, subset = group == "S")

In [ ]:
# This function is used to get or set the identifiers of an object. The new columm is named "celltype"
Idents(testdata) <- "celltype"

In [ ]:
# Normalize the single-cell RNA-seq data using Seurat
testdata <- Seurat::NormalizeData(testdata, verbose = FALSE)

In [ ]:
# Display the structure of the processed dataset
testdata %>% dplyr::glimpse()

## 运行 LIANA


运行 LIANA 时，可以选择它支持的任意方法。在这个示例中，我们使用 [CellPhoneDB](https://www.nature.com/articles/s41596-020-0292-x) 的实现。

`liana_wrap` 函数会调用多种方法，每种方法都会使用所提供的资源。如果没有指定具体方法，`liana_wrap` 会执行 LIANA 中实现的所有方法。此外，默认会使用 `consensus` 资源。


In [ ]:
scData <- as.SingleCellExperiment(testdata, assay = "RNA")

In [ ]:
cpdb_result <- liana_wrap(scData, # This function of the LIANA package is used to perform cell-cell interaction analyses using different methods
                          method = 'cellphonedb',
                          resource = c('CellPhoneDB'),                 # Specifies that the CellPhoneDB method will be used
                          permutation.params = list(nperms=100,        # Defines the permutation parameters. nperms=100: Number of permutations
                                                    parallelize=FALSE, # Indicates whether execution should be parallelized
                                                    workers=4),        # Number of workers to use if execution were parallelized
                          idents_col = "cluster",
                          expr_prop=0.05)   # Minimum proportion of expression to consider an interaction as valid

In [ ]:
# Display the structure of the CellPhoneDB interaction results
dplyr::glimpse(cpdb_result)

如果希望同时使用多种方法运行 LIANA，可以在 `method` 参数中指定目标方法。在这个示例中，我们将使用 CellPhoneDB、NATMI、SingleCellSignalR（sca）以及 logFC 方法。


In [ ]:
# Run a more complex interaction analysis using multiple methods
complex_test <- liana_wrap(scData,
                           method = c('cellphonedb', 'natmi', 'sca', 'logfc'),
                           resource = c('CellPhoneDB'),  # Use CellPhoneDB resource
                           idents_col = "cluster") 

In [ ]:
# Display the structure of the complex interaction results
dplyr::glimpse(complex_test)

LIANA 的一个关键特性是可以对所有用于细胞间通讯分析的方法预测结果计算 consensus ranking。通过 `liana_aggregate()` 函数，可以整合上一步中所有方法得到的结果。


In [ ]:
# Aggregate interaction results across multiple methods
liana_consensus <- complex_test %>% liana_aggregate()

In [ ]:
# Display the structure of the aggregated interaction results
dplyr::glimpse(liana_consensus)

## 可视化与解释


Dotplot 可用于直观解释 sender-receiver 细胞对之间的重要 ligand-receptor 配对。

这里我们先预处理 CellPhoneDB 的结果，然后绘图。分析中会应用筛选条件，只使用显著结果（P-value < 0.05）。


In [ ]:
cpdb_int <- cpdb_result %>%
  # Only keep interactions with p-val <= 0.05
  filter(pvalue <= 0.05) %>% # This reflects interactions `specificity`
  rank_method(method_name = "cellphonedb", mode = "magnitude") %>% # Then rank according to `magnitude` (lr_mean in this case)
  distinct_at(c("ligand.complex", "receptor.complex")) %>%         # Keep top 20 interactions (regardless of cell type)
  head(20)  

In [ ]:
# Options(repr.plot.height = 12, repr.plot.width = 9)
# Plot cell-cell interactions using LIANA's dotplot function
scPlot <- cpdb_result %>%  
          inner_join(cpdb_int, # Keep only the interactions of interest
                    by = c("ligand.complex", "receptor.complex")) %>%  # Invert size (low p-value/high specificity = larger dot size), add a small value to avoid Infinity for 0s
          mutate(pvalue = -log10(pvalue + 1e-10)) %>%                  # Transforms the p-value by taking the negative log10
          liana_dotplot(source_groups = c("Epithelial"),               # Creates a dot plot for the specified source and target groups, and specifies the source group
                        target_groups = c("Macrophages", "NK", "B", "T", "Neutrophil"), # Specifies the target groups.
                        specificity = "pvalue",  # Uses the p-value for dot size specification.
                        magnitude = "lr.mean",   # Uses the mean log ratio for the dot color
                        show_complex = TRUE,
                        size.label = "-log10(p-value)") + theme(axis.text.x = element_text(angle = 90))
scPlot
# Save the plot as an image file
ggsave("01-liana_dotplot.png", plot = scPlot, bg = "white", dpi = 600, width = 16, height = 9)

类似地，我们也可以探索 consensus 结果。


In [ ]:
# Options(repr.plot.height = 12, repr.plot.width = 9)
# Plot top 20 interactions from aggregated LIANA results
scPlot <- liana_consensus %>%
          liana_dotplot(source_groups = c("Macrophages"),
                        target_groups = c("Macrophages", "NK", "B", "T", "Neutrophil"),
                        ntop = 20) + theme(axis.text.x = element_text(angle = 90))
scPlot
# Save the plot
ggsave("02-liana_dotplot.png", plot = scPlot, bg = "white", dpi = 600, width = 16, height = 9)

还可以计算细胞整体的潜在通讯能力。这里可以统计显著或重要相互作用的数量，然后通过热图可视化。

类似地，我们可以比较 CellPhoneDB 与 consensus 的结果。


In [ ]:
# Filter interactions with p-value ≤ 0.05 and plot frequency heatmap
liana_trunc <- cpdb_result %>% filter(pvalue <= 0.05)

# Generate and save heatmap
# png("03-heat_freq.png", bg = "white")
heat_freq(liana_trunc)
# dev.off()

In [ ]:
# Filter consensus interactions with aggregate rank ≤ 0.01 and plot heatmap
liana_trunc <- liana_consensus %>% filter(aggregate_rank <= 0.01)

# Generate and save heatmap
# png("04-heat_freq.png", bg = "white")
heat_freq(liana_trunc)
# dev.off()

## 思考题：

- 任选两种方法，它们的结果差异有多大？

- 为什么会出现这种差异？


In [ ]:
# List all functions available in the LIANA package
ls("package:liana")